In [1]:
import os

In [2]:
%pwd

'e:\\KrishNair\\Kidney_Disease_Classification_Deep_Learning_Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\KrishNair\\Kidney_Disease_Classification_Deep_Learning_Project'

In [5]:
import dagshub
dagshub.init(repo_owner='divyadarshan.dsa', repo_name='Kidney-Disease-Classification-Deep-Learning-Project', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as divyadarshan.dsa

Initialized MLflow to track repo "divyadarshan.dsa/Kidney-Disease-Classification-Deep-Learning-Project"

Repository divyadarshan.dsa/Kidney-Disease-Classification-Deep-Learning-Project initialized!

2026/09/10 02:35:40 INFO mlflow.tracking._tracking_service.client: 🏃 View run fearless-yak-586 at: https://dagshub.com/divyadarshan.dsa/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0/runs/9b5d05a289bc499f85d1865416ad627b.
2026/09/10 02:35:40 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/divyadarshan.dsa/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0.


In [6]:
%pip install dagshub

In [7]:
%pip install mlflow

Note: you may need to restart the kernel to use updated packages.


In [8]:
import tensorflow as tf

In [9]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [11]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [12]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [13]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney-ct-scan-image",
            mlflow_uri="https://dagshub.com/entbappy/Kidney-Disease-Classification-MLflow-DVC.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [14]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [15]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1. / 255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()

        # Changed model.evaluate() to self.model.evaluate()
        self.score = self.model.evaluate(self.valid_generator)

        self.save_score()

    def save_score(self):
        scores = {
            "loss": self.score[0],
            "accuracy": self.score[1]
        }

        save_json(
            path=Path("scores.json"),
            data=scores
        )

    def log_into_mlflow(self):

        with mlflow.start_run():

            mlflow.log_params(self.config.all_params)

            mlflow.log_metrics(
                {
                    "loss": self.score[0],
                    "accuracy": self.score[1]
                }
            )

            # Log the model without registering it
            mlflow.keras.log_model(
                self.model,
                "model"
            )

In [16]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

[2026-09-10 02:38:53,484: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-10 02:38:54,430: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-10 02:38:54,446: INFO: common: created directory at: artifacts]
Found 2207 images belonging to 2 classes.
138/138 [==============================] - 701s 5s/step - loss: 16.6399 - accuracy: 0.6901
[2026-09-10 02:50:38,229: INFO: common: json file saved at: scores.json]


2026/09/10 02:50:55 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: C:\Users\PC\AppData\Local\Temp\tmputxod0ku\model\data\model\assets
[2026-09-10 02:51:03,633: INFO: builder_impl: Assets written to: C:\Users\PC\AppData\Local\Temp\tmputxod0ku\model\data\model\assets]


2026/09/10 02:51:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/09/10 02:52:44 INFO mlflow.tracking._tracking_service.client: 🏃 View run classy-ox-74 at: https://dagshub.com/divyadarshan.dsa/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0/runs/f00adffcf48e4e8dabe4d42f6d4a397a.
2026/09/10 02:52:44 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/divyadarshan.dsa/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0.
